# Chapter 9 — Ontologies and Natural Languages
### Notebook 2 · Multilingual ontologies

*Book reference: Section 9.1*

The axioms are language-independent; the labels are not. That single separation is what lets one ontology serve many languages — and it is also where half-translated ontologies hide.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch09_toolkit as ch9
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. One ontology, several lexicons

Note what does **not** change between languages: the terms. `Giraffe` is the ontology's identifier in every language; only its surface form differs.

In [ ]:
rows = [{'term': term, **labels} for term, labels in ch9.LEXICON.items()]
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
axiom = ch9.Axiom('Lion', 'some', 'Herbivore', 'eats')
print('axiom:', axiom, '\n')
for language in ch9.LANGUAGES:
    print(f'  {language}: {ch9.verbalise(axiom, language)}')

## 2. The round trip works in every language

Each language has its own templates and its own reverse lexicon, so fidelity is checkable per language.

In [ ]:
rows = []
for language in ch9.LANGUAGES:
    ok = sum(ch9.round_trips(a, language)['ok'] for a in ch9.SAMPLE_AXIOMS)
    rows.append({'language': language, 'round-trips': f'{ok}/{len(ch9.SAMPLE_AXIOMS)}'})
print(pd.DataFrame(rows).to_string(index=False))
assert all(ch9.round_trips(a, l)['ok']
           for a in ch9.SAMPLE_AXIOMS for l in ch9.LANGUAGES)

## 3. Coverage: what a language is missing

Translation is rarely complete. `label_for` falls back to the identifier, so an unlexicalised term renders as its IRI fragment — quietly.

In [ ]:
for language in ch9.LANGUAGES:
    print(' ', ch9.lexicon_coverage(language))

In [ ]:
partial = {k: dict(v) for k, v in ch9.LEXICON.items()}
for term in ['Herbivore', 'Leaf', 'isPartOf']:
    partial[term].pop('nl', None)          # simulate an incomplete translation
original = ch9.LEXICON
ch9.LEXICON = partial
try:
    print('coverage now:', ch9.lexicon_coverage('nl')['coverage'])
    print('missing     :', ch9.lexicon_coverage('nl')['missing'])
    print()
    for a in ch9.SAMPLE_AXIOMS[:3]:
        print('  ', ch9.verbalise(a, 'nl'))
finally:
    ch9.LEXICON = original

> **`Elke giraf is een Herbivore.`** — Dutch grammar, an English noun, and no error anywhere. This is what a half-translated ontology looks like in production: mostly right, quietly wrong, and impossible to spot without measuring coverage per language.

## 4. Fidelity does not imply translation

Worse: the sentence above still **round-trips**. The parser falls back to returning an unknown label unchanged, so the axiom is recovered exactly. An exact metric reports success on a sentence that is not really Dutch.

In [ ]:
sentence = 'Elke giraf is een Herbivore.'
recovered = ch9.parse_cnl(sentence, 'nl')
print('sentence :', sentence)
print('recovered:', recovered)
print('faithful :', recovered.key() == ch9.SAMPLE_AXIOMS[0].key())
print()
print('but is it Dutch?')
print(json.dumps(ch9.uses_lexicon_labels(sentence, ch9.SAMPLE_AXIOMS[0], 'nl'), indent=1))

In [ ]:
good = ch9.verbalise(ch9.SAMPLE_AXIOMS[0], 'nl')
print('properly localised:', good)
print(json.dumps(ch9.uses_lexicon_labels(good, ch9.SAMPLE_AXIOMS[0], 'nl'), indent=1))
print('\nTwo different checks, two different questions: "did the meaning\n'
      'survive?" and "is this the requested language?". Neither implies the\n'
      'other, which is why the Chapter 9 metric scores both.')

### Exercise 2.1 — Add a fourth language

Add French labels for four terms, verbalise an axiom, and report coverage. State what would break if you added the labels but not a template set.

> **Hint.** `verbalise` looks up `TEMPLATES[language]`, not just the lexicon.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 2.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
original = {k: dict(v) for k, v in ch9.LEXICON.items()}
try:
    ch9.LEXICON['Giraffe']['fr'] = 'girafe'
    ch9.LEXICON['Herbivore']['fr'] = 'herbivore'
    ch9.LEXICON['Lion']['fr'] = 'lion'
    ch9.LEXICON['Animal']['fr'] = 'animal'
    print('fr coverage:', ch9.lexicon_coverage('fr')['coverage'])
    print('fr missing :', ch9.lexicon_coverage('fr')['missing'][:5], '...')
    print('\nlabels resolve:', ch9.label_for('Giraffe', 'fr'),
          '/', ch9.label_for('Herbivore', 'fr'))
    try:
        ch9.verbalise(ch9.SAMPLE_AXIOMS[0], 'fr')
        print('verbalised in French')
    except KeyError as exc:
        print('\nverbalisation FAILS:', repr(exc))
        print('A lexicon is not enough. Verbalisation needs TEMPLATES too, and\n'
              'templates encode grammar -- article agreement, word order,\n'
              'inflection. Adding a language is a grammar job, not a glossary job,\n'
              'which is the practical reason multilingual ontologies stall.')
finally:
    ch9.LEXICON.clear(); ch9.LEXICON.update(original)

### Exercise 2.2 — Measure how much translation is really done

Report, per language, the fraction of the sample axioms that verbalise using **only** that language's labels.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 2.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
rows = []
for language in ch9.LANGUAGES:
    localised = sum(
        ch9.uses_lexicon_labels(ch9.verbalise(a, language), a, language)['ok']
        for a in ch9.SAMPLE_AXIOMS)
    rows.append({'language': language,
                 'fully localised': f'{localised}/{len(ch9.SAMPLE_AXIOMS)}',
                 'lexicon coverage': ch9.lexicon_coverage(language)['coverage']})
print(pd.DataFrame(rows).to_string(index=False))
assert all(r['lexicon coverage'] == 1.0 for r in rows)
print('\nWith a complete lexicon every axiom localises fully. The number to watch\n'
      'in a real project is the FIRST column: coverage of the vocabulary is not\n'
      'the same as coverage of the sentences people actually generate, because\n'
      'a single missing term spoils every axiom that mentions it.')